# Inverse problem: reaction rate constants from concentration data

A staple of chemical kinetics is the **consecutive first-order reaction** $A\to B\to C$:
$$\dot A=-k_1 A,\qquad \dot B=k_1 A-k_2 B,\qquad \dot C=k_2 B,\qquad A(0)=1,\ B(0)=C(0)=0.$$
The **rate constants $k_1,k_2$ are unknown**. Given noisy measurements of the three concentrations, recover them — the everyday task of the experimental chemist, who observes concentrations and must infer the kinetics.

As in the wave-speed inverse problem, we simply **promote $k_1,k_2$ to trainable parameters**: one network outputs $(A,B,C)(t)$, two extra scalars are the rates, and the very same Adam step that fits the network adjusts them. There is no outer loop and no repeated forward solve.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

K1T, K2T, A0, T = 1.0, 0.5, 1.0, 8.0            # TRUE rate constants (unknown to the PINN)
def exact(t):
    A = A0*np.exp(-K1T*t)
    B = A0*K1T/(K2T-K1T)*(np.exp(-K1T*t) - np.exp(-K2T*t))
    return np.stack([A, B, A0 - A - B])

# synthetic experiment: 40 noisy concentration measurements
ND = 40; td = np.sort(np.random.rand(ND))*T
obs = exact(td) + 0.03*np.random.randn(3, ND)          # 3% Gaussian noise
td_t  = torch.tensor(td,  dtype=torch.float32, device=device).reshape(-1,1)
obs_t = torch.tensor(obs.T, dtype=torch.float32, device=device)

def dt(f, t): return torch.autograd.grad(f, t, torch.ones_like(f), create_graph=True)[0]

In [ ]:
net = nn.Sequential(nn.Linear(1,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(),
                    nn.Linear(64,64), nn.Tanh(), nn.Linear(64,3)).to(device)
logk = torch.nn.Parameter(torch.log(torch.tensor([0.3, 0.3], device=device)))  # deliberately wrong start
opt  = torch.optim.Adam(list(net.parameters()) + [logk], 3e-3)
y0   = torch.tensor([A0, 0.0, 0.0], dtype=torch.float32, device=device)
def trial(t): return y0 + t*net(t/T)                    # hard-enforces A0, 0, 0 at t=0

hist = []; EP = 15000; t0 = time.perf_counter()
for e in range(EP):
    if e == int(0.6*EP):
        for g in opt.param_groups: g['lr'] = 5e-4
    opt.zero_grad()
    k1, k2 = torch.exp(logk)                            # positive by construction
    t = (torch.rand(1024,1, device=device)*T).requires_grad_(True)
    y = trial(t); A, B, C = y[:,0:1], y[:,1:2], y[:,2:3]
    res  = ((dt(A,t) + k1*A)**2 + (dt(B,t) - k1*A + k2*B)**2 + (dt(C,t) - k2*B)**2).mean()
    data = ((trial(td_t) - obs_t)**2).mean()
    (res + 10.0*data).backward(); opt.step()
    if e % 200 == 0: hist.append((e,) + tuple(torch.exp(logk).detach().cpu().numpy()))
if device.type == 'cuda': torch.cuda.synchronize()
k1, k2 = [float(x) for x in torch.exp(logk).detach().cpu().numpy()]
print(f'training time: {time.perf_counter()-t0:.0f} s')
print(f'recovered  k1 = {k1:.4f}  (true {K1T})')
print(f'recovered  k2 = {k2:.4f}  (true {K2T})')

In [ ]:
tg = np.linspace(0, T, 300); ex = exact(tg)
with torch.no_grad():
    yp = trial(torch.tensor(tg, dtype=torch.float32, device=device).reshape(-1,1)).cpu().numpy().T
hist = np.array(hist)
lab = ['A', 'B', 'C']; col = ['tab:blue', 'tab:orange', 'tab:green']
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for k in range(3):
    ax[0].plot(tg, ex[k], color=col[k], lw=2.4, alpha=.5, label=f'{lab[k]} exact')
    ax[0].plot(tg, yp[k], '--', color=col[k], lw=1.5)
    ax[0].plot(td, obs[k], '.', color=col[k], ms=4, alpha=.5)
ax[0].plot([], [], 'k--', label='PINN'); ax[0].plot([], [], 'k.', label='noisy data')
ax[0].set_xlabel('time'); ax[0].set_ylabel('concentration'); ax[0].legend(fontsize=8, ncol=2)
ax[0].grid(alpha=.3); ax[0].set_title(r'A$\to$B$\to$C: reconstructed concentrations')
ax[1].plot(hist[:,0], hist[:,1], color='tab:blue', label='$k_1$ (learned)')
ax[1].plot(hist[:,0], hist[:,2], color='tab:red',  label='$k_2$ (learned)')
ax[1].axhline(K1T, color='tab:blue', ls=':', lw=1.2); ax[1].axhline(K2T, color='tab:red', ls=':', lw=1.2)
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('rate constant'); ax[1].legend(fontsize=9)
ax[1].grid(alpha=.3); ax[1].set_title(f'Rate constants converge (k1={k1:.2f}, k2={k2:.2f})')
plt.tight_layout(); plt.show()

From a deliberately wrong start ($k_1=k_2=0.3$) the optimiser recovers both rate constants to a couple of per cent — the residual limit set by the 3% measurement noise — while reconstructing the concentration profiles. The unknown constants are learned *through* the physics, in the same training that fits the data, with no forward solve ever repeated.